<a href="https://colab.research.google.com/github/cpython-projects/da_27_07_2026/blob/main/lesson_17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Кореляція, статистична значущість і довірчі інтервали

**Навіщо це потрібно.** Досі ми вміли описати одну змінну (середнє, медіана, розподіл). Але бізнес-питання майже завжди про **зв'язок двох змінних**: чи впливає площа на ціну, чи варта вкладень якість матеріалів, наскільки точна наша оцінка середньої ціни. У цій лекції розглядаємо три коефіцієнти кореляції (Pearson, Spearman, Kendall), статистичну значущість (p-value) і довірчі інтервали — інструменти, якими аналітик відповідає на ці питання не "на око", а числом з обґрунтуванням.

Будемо працювати з тим самим датасетом будинків і розбирати **5 реальних бізнес-питань** одне за одним.

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/cpython-projects/python_da_17_03_26/refs/heads/main/kc_house_data.csv')

In [ ]:
df.head()

In [ ]:
df.info()

# Питання 1

**Наскільки вигідно інвестувати в збільшення площі будинку?**
**Чи кожні додаткові 10 м² реально дають відчутний приріст ціни?**

## Що таке кореляція

### Інтуїтивне визначення

Кореляція — це міра того, **наскільки дві змінні змінюються разом**.

Не:

> "X викликає Y"

А:

> "Коли X зростає, що зазвичай відбувається з Y?"

---

### Приклади

* Чим більша реклама → тим більші продажі?
* Чим більше досвіду → тим вища зарплата?
* Чим більша площа → тим дорожче житло?

---

### Діапазон і сила зв'язку

Кореляція ∈ [-1, 1]. **Знак** показує напрямок зв'язку, а **сила** зв'язку визначається модулем `|r|` — і таблиця нижче саме про це: інтерпретація однакова для `r = 0.6` і для `r = -0.6`, різниця лише в напрямку.

| r | Інтерпретація |
|---|---|
| 0.00–0.10 | Практично немає зв'язку |
| 0.10–0.30 | Дуже слабкий |
| 0.30–0.50 | Слабкий / помірний |
| 0.50–0.70 | Помірний / сильний |
| 0.70–0.90 | Сильний |
| 0.90–1.00 | Дуже сильний / майже ідеальний |

**Знак кореляції — це теж бізнес-інформація:**
* позитивна (r > 0): чим більше X → тим більше Y
  приклад: площа → ціна
* негативна (r < 0): чим більше X → тим менше Y
  приклад: вік будинку → ціна

### Кореляція Pearson — лінійна залежність

Кореляція Pearson відповідає на питання:

> **Чи можна описати зв'язок між X і Y майже прямою лінією?**

* Позитивний r → X↑ → Y↑
* Негативний r → X↑ → Y↓
* r близько 0 → прямої залежності майже немає

#### Умови, коли допустимо використовувати:

* Обидві змінні **кількісні (interval/ratio)**
* Зовнішній вигляд графіку **наближено лінійний**
* Немає сильних викидів, або їх мало
* Мета — **лінійний тренд** (Y ~ a·X + b)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.scatter(df['sqft_living'], df['price'], alpha=0.3)
plt.xlabel('Living area (sqft)')
plt.ylabel('Price')
plt.title('Price vs Living Area')
plt.show()

In [ ]:
df[['price', 'sqft_living']].corr(method='pearson')

`r ≈ 0.70` — сильний позитивний зв'язок: більша площа майже завжди означає вищу ціну, і залежність добре наближається прямою лінією (це видно й на графіку вище).

Площа — важливий фактор, але недостатній сам по собі: `r = 0.70` означає, що площа пояснює далеко не всю варіацію ціни (`r² ≈ 0.49`, тобто ≈ 49% розкиду ціни). Для точних прогнозів потрібно також враховувати інші характеристики.

# Питання 2

**Чи варто робити більше кімнат, чи після певного моменту це не має сенсу?**

## Кореляція Spearman — монотонна залежність

Spearman відповідає на питання:

> **Чи змінюється Y разом із X в одному напрямку, навіть якщо не по прямій?**

* Позитивний ρ → X↑ → Y↑
* Негативний ρ → X↑ → Y↓
* ρ близько 0 → монотонної залежності майже немає

---

#### Умови, коли допустимо використовувати:

* Обидві змінні **кількісні (interval/ratio)** або **рангові (ordinal)**
* Залежність може бути **нелінійною, але монотонною**
* Є **викиди**, які можуть спотворювати Pearson
* Мета — оцінити **монотонний тренд**, а не лінійність

In [ ]:
df[['price', 'bedrooms']].corr(method='spearman')

`ρ ≈ 0.34` — слабкий/помірний зв'язок. Кількість спалень впливає на ціну (більше спалень загалом означає дорожче), але це значно слабший фактор, ніж площа (`ρ` тут набагато нижчий за `r = 0.70` з питання 1) — і це узгоджується з інтуїцією: двоє будинків з однаковою кількістю спалень можуть сильно відрізнятись за площею й ціною. "Після певного моменту" ефект не зникає повністю, але явно послаблюється.

# Питання 3

Чи виправдано вкладатися в дорогі матеріали та преміум-ремонт?
Чи це реально конвертується в вищу ціну продажу?

### Kendall Tau — порядок/ранги

Kendall відповідає на питання:

> **Наскільки збігаються порядки між X і Y?**

* Позитивний τ → більші X співпадають із більшими Y
* Негативний τ → більші X співпадають із меншими Y
* τ близько 0 → порядок майже не збігається

---

# Що означає «порядки»

«Порядок» = **хто вище, хто нижче, хто попереду, хто позаду.**
Не важливо *наскільки*, важливо **хто кращий від кого**.

| Студент | Оцінка з математики | Оцінка з фізики |
| ------- | ------------------- | --------------- |
| A       | 90                  | 88              |
| B       | 85                  | 80              |
| C       | 70                  | 75              |
| D       | 60                  | 65              |
| E       | 50                  | 40              |

Тут порядок майже однаковий:

Math: A > B > C > D > E
Physics: A > B > C > D > E

**Порядки збігаються ідеально.**
Kendall ≈ 1.

---

#### Умови, коли допустимо використовувати:

* Дані **рангові (ordinal)** або **кількісні**, де важливий порядок, а не точна величина
* Є багато однакових значень (**ties**) — типова ситуація для дискретних оцінкових шкал, як наш `grade` (цілі значення 1–13, де одне й те саме значення зустрічається тисячі разів)
* Мета — оцінити **узгодженість порядків**, а не силу лінійного тренду

Kendall особливо корисний саме через ties: Pearson і Spearman формально теж рахуються за наявності однакових значень, але Kendall спроєктований враховувати їх коректніше.

У датасеті є дві різні характеристики, які легко сплутати:

* `grade` — оцінка **якості будівництва й використаних матеріалів** (1–13);
* `condition` — оцінка **стану/догляду** за будинком (1–5), тобто ближче саме до "ремонту" в розмовному сенсі.

Перевіримо обидві, а не лише одну — питання ж прямо каже "дорогі матеріали **та** ремонт".

In [ ]:
df[['price', 'grade']].corr(method='kendall')

In [ ]:
df[['price', 'condition']].corr(method='spearman')

Результат несподіваний:

* `grade` (якість матеріалів/будівництва) ↔ `price`: `τ ≈ 0.53` — помірний/сильний зв'язок.
* `condition` (стан будинку, тобто ближче до "ремонту") ↔ `price`: `ρ ≈ 0.02` — зв'язку практично немає.

**Висновок складніший, ніж здавалося спочатку.** Дорогі матеріали та вища категорія будівництва (`grade`) дійсно сильно конвертуються у вищу ціну. А от сам по собі хороший **стан/догляд** (`condition`) — майже ніяк не пов'язаний з ціною в цих даних: покупці на цьому ринку платять за те, **з чого й як побудований** будинок, а не за те, наскільки доглянутим він здається на момент продажу. "Преміум-ремонт окупається" — правда лише в тій частині, де йдеться про якість матеріалів і рівень будівництва, а не про косметичний стан.

### Практичний алгоритм вибору

#### 1) Подивись на шкалу

* **Ratio/Interval** → Pearson або Spearman
* **Ordinal** → Spearman або Kendall

---

#### 2) Подивись на графік

* Якщо точки *виглядають по прямій* → Pearson
* Якщо тренд *монотонний, але не пряма* → Spearman
* Якщо багато однакових значень або порядки важливі → Kendall

---

#### 3) Подивись на розподіл

* Сильно *не нормальний, важкі хвости,* або *викиди* → Spearman / Kendall
* Нормальний/виражений лінійний тренд → Pearson

## Бонус: автоматичний EDA-звіт

Усе, що вище, можна рахувати вручну по парах колонок. Але для швидкого **первинного огляду** нового датасету існують бібліотеки, що одразу будують звіт по всіх колонках і всіх парних кореляціях — `ydata-profiling` одна з найпопулярніших.

> ⚠️ Звіт по 21 тис. рядків і 21 колонці може рахуватись кілька хвилин, а сама бібліотека чутлива до версій pandas/numpy в середовищі. Це інструмент для **першого побіжного огляду**, а не заміна цілеспрямованого аналізу з конкретним бізнес-питанням, як вище.

In [ ]:
!pip install ydata_profiling

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, minimal=True)
profile.to_file("report.html")

`minimal=True` вимикає найважчі обчислення (деякі взаємні кореляції й тести) і суттєво прискорює звіт на датасеті такого розміру — корисно знати цей параметр, інакше перший запуск на новому датасеті може "зависнути" на незрозуміло довгий час.

# Розрахунок кореляції через scipy.stats

* Повертає **коефіцієнт кореляції** і **p-value**
* Можна вибрати потрібний метод (`pearsonr`, `spearmanr`, `kendalltau`)
* **Плюс:** отримуємо статистичну значущість
* **Мінус:** треба робити для **пар змінних окремо**, не таблицею

---

## Що таке p-value у кореляції

* **p-value** показує, наскільки отримане значення кореляції могло виникнути **випадково**, якщо насправді **зв'язку немає**
* Інтерпретація:

| p-value | Інтерпретація |
|---|---|
| < 0.05 | кореляція статистично значуща (висока ймовірність, що зв'язок реальний) |
| ≥ 0.05 | кореляція **не є статистично значущою** (може бути випадковою) |

## ⚠️ Значущість — це не те саме, що важливість

На великих датасетах (у нас 21 613 рядків) **майже будь-яка ненульова кореляція** виявиться "статистично значущою", навіть якщо вона нічого не варта для бізнесу. Перевірмо на `condition`, яку щойно визнали практично незв'язаною з ціною:

In [ ]:
from scipy import stats

stats.pearsonr(df.price, df.condition)

`r ≈ 0.036` (практично немає зв'язку за нашою ж таблицею), а `p-value ≈ 9×10⁻⁸` — набагато менше 0.05, тобто формально "дуже значуще". Це не протиріччя: p-value лише каже, що зв'язок, швидше за все, **не дорівнює нулю рівно**, а не що він **сильний**. На малій вибірці (десятки, сотні рядків) слабкий зв'язок цілком може виявитись незначущим — а на десятках тисяч рядків навіть крихітний, нецікавий для бізнесу зв'язок майже завжди "пройде" тест на значущість.

**Практична порада:**

* **Якщо треба швидко оцінити матрицю кореляцій** → `df.corr()`
* **Якщо потрібна статистична значущість** → `scipy.stats`
* **Головне правило:** спочатку дивіться на силу зв'язку (`|r|`), і лише потім — чи вона значуща. Значущість без сили — малоінформативна на великих даних.

# Питання 4

**«Що сильніше впливає на ціну: рік побудови чи якість будинку?»**

In [ ]:
corr, pvalue = stats.pearsonr(df.price, df.sqft_living)
corr, pvalue

In [ ]:
df.columns.to_list()

In [ ]:
df[['price', 'yr_built', 'grade']].corr(method='spearman')

In [ ]:
stats.spearmanr(df.price, df.grade)

`price` ↔ `grade`: `ρ ≈ 0.66`. `price` ↔ `yr_built`: `ρ ≈ 0.10` — у 6 разів слабше.

Ціну формує не вік будинку, а його якість.
Рік побудови важливий лише настільки, наскільки він пов'язаний із якістю (новіші будинки частіше мають вищий `grade`, звідси й невелика, але ненульова кореляція `yr_built` із ціною).

# Питання 5

**В якому діапазоні реально знаходиться середня ціна будинку в нашому сегменті ринку?**

In [ ]:
df.price.mean()

## Середня й типова ціна — це НЕ одне й те саме

Перш ніж рахувати довірчі інтервали, варто чітко розвести два поняття, які легко сплутати:

* **Середня (арифметична) ціна** — сума всіх цін, поділена на їх кількість. Рахується як `df.price.mean()` → **540 088 usd**.
* **Типова ціна** — ціна "звичайного", найпоширенішого будинку. Найчастіше за типову беруть **медіану** → `df.price.median()` → **450 000 usd**.

Різниця між ними — **майже 90 000 usd**, і це не випадковість. На ринку нерухомості є невелика кількість дуже дорогих будинків (мільйони доларів), і кожен такий будинок "тягне" арифметичне середнє вгору набагато сильніше, ніж один звичайний будинок тягне його вниз. Медіана ж на це нечутлива — їй байдуже, продався сусідній будинок за 800 000 чи за 8 000 000, важливо тільки, що він десь "вище середини".

**Тому в бізнесі треба явно казати, яке саме число ви маєте на увазі.** "Середня ціна 540 тис." і "типова ціна 450 тис." — це обидва правильні, перевірені числа про той самий ринок, просто відповіді на різні питання:

* "Скільки грошей в сумі приносить продаж N будинків?" → питання про **середнє** (сума = середнє × N).
* "За скільки продається звичайний, нічим не примітний будинок?" → питання про **типову ціну / медіану**.

Нижче ми побудуємо довірчий інтервал **для середнього** — і важливо пам'ятати, що це інтервал саме навколо числа 540 088, а не навколо медіани 450 000. Те, що ці два числа так сильно відрізняються одне від одного, — це нормально для скошеного розподілу, а не помилка в розрахунках.

## Що таке довірчий інтервал?

Довірчий інтервал — це **діапазон значень**, який з певним рівнем впевненості (наприклад, 95%) **містить істинне середнє** генеральної сукупності.

Простими словами:

* Якщо ми обчислюємо середню ціну будинків по вибірці, ми не знаємо точне середнє всіх будинків.
* ДІ дає нам **інтервал**, де це середнє з великою ймовірністю знаходиться.

**Приклад:**

* Середня ціна будинку = 700 000, 95% ДІ = (680 000, 720 000)
* Ми можемо сказати: «Істинне середнє серед всіх будинків, ймовірно, знаходиться між 680 000 та 720 000»

---

## Чому важливо для аналітика?

* ДІ дозволяє **оцінити точність метрики** (середнє, відсотки, LTV і т.д.)
* Може допомогти **порівнювати сегменти** (наприклад, райони чи категорії будинків)
* Використовується для **звітності та обґрунтування рішень**

---

## Формула для довірчого інтервалу середнього

Якщо у нас вибірка `x1, x2, …, xn` з середнім `x̄` і стандартним відхиленням `s`, ДІ обчислюється як:

$$
CI = x̄ \pm z \cdot \frac{s}{\sqrt{n}}
$$

* `z` — критичне значення для рівня довіри (для 95% → 1.96)
* `s` — стандартне відхилення вибірки
* `n` — розмір вибірки

> 💡 Строго кажучи, `z = 1.96` коректний, коли справжнє стандартне відхилення генеральної сукупності відоме, або коли `n` дуже велике (тоді t-розподіл Стʼюдента практично збігається з нормальним). При малих вибірках (десятки спостережень) правильніше брати квантиль **t-розподілу**. У нас `n = 21613`, тож 1.96 — цілком обґрунтоване наближення.

### Що таке t-розподіл і коли він потрібен

Формула ДІ використовує `s` — стандартне відхилення, пораховане **по вибірці**, а не справжнє стандартне відхилення всієї генеральної сукупності (яке нам ніколи не відоме). Ця заміна вносить додаткову невизначеність: ми оцінюємо розкид даних по тих самих даних, по яких рахуємо середнє.

**t-розподіл Стʼюдента** — це розподіл, який враховує саме цю додаткову невизначеність. Він схожий на нормальний (так само дзвоноподібний і симетричний), але має **товщі хвости** — тобто "дозволяє" трохи більший розкид, ніж нормальний, і тому дає **ширший** (чесніший) довірчий інтервал. Наскільки товщі хвости — залежить від **кількості ступенів свободи**, яка дорівнює `n - 1`: чим менша вибірка, тим товщі хвости й тим сильніше t-розподіл відрізняється від нормального. Коли `n` велике, оцінка `s` стає дуже точною, додаткова невизначеність зникає, і t-розподіл практично зливається з нормальним.

Порівняймо критичне значення для 95% ДІ (аналог нашого `z = 1.96`) з t-розподілу при різній кількості спостережень:

In [ ]:
from scipy import stats as stats_check

for sample_n in [5, 10, 30, 100, 1000, 21613]:
    t_value = stats_check.t.ppf(0.975, df=sample_n - 1)
    print(f'n={sample_n:>6}: t = {t_value:.4f}')

print(f'нормальне наближення (z): {stats_check.norm.ppf(0.975):.4f}')

При `n = 5` t-критичне значення (`≈2.78`) помітно більше за `z = 1.96` — довірчий інтервал на такій малій вибірці вийшов би майже на 40% ширшим, ніж якщо (помилково) скористатися нормальним наближенням. А вже при `n = 1000` різниця зникає в третьому знаку після коми. При наших `n = 21613` використання `z = 1.96` замість точного t-значення (`≈1.9601`) — абсолютно безпечне спрощення.

**Правило на практиці:** вибірка з десятків-сотень спостережень → рахуйте ДІ через t-розподіл (`scipy.stats.t.interval(...)`, враховуючи `df = n - 1`). Вибірка з тисяч і більше спостережень → різниця з нормальним наближенням настільки мала, що простіша формула з `z = 1.96` цілком прийнятна.

## А чи можна взагалі рахувати ДІ по "сирій" ціні?

Ціна явно не нормально розподілена — подивимось:

In [ ]:
df.price.hist(bins=100)

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

stats.probplot(df.price, dist='norm', plot=plt)
plt.title('QQ-plot: price (сирі дані)')
plt.show()

Гістограма сильно скошена праворуч, а QQ-plot чітко відхиляється від прямої на правому хвості (дорогі будинки) — `price` явно не нормальний розподіл.

**І все ж формула ДІ вище працює навіть для таких даних.** Причина — **центральна гранична теорема (CLT)**: вона каже, що розподіл *вибіркового середнього* (а не самих даних!) наближається до нормального, коли `n` достатньо велике, **незалежно від форми розподілу вихідних даних**. У нас `n = 21613` — величезна вибірка, тож формулу можна застосовувати напряму до `price`, без жодних трансформацій:

In [ ]:
import numpy as np

mean_price = df.price.mean()
std_price = df.price.std()
n = len(df)

ci_price = stats.norm.interval(0.95, loc=mean_price, scale=std_price / np.sqrt(n))
ci_price

Це і є відповідь на питання 5: **справжня середня ціна по всьому ринку, ймовірно, лежить у цьому вузькому діапазоні (±~1% від вибіркового середнього)** — і це коректний результат, попри те, що сам розподіл `price` далекий від нормального.

## Навіщо тоді взагалі log-трансформація?

CLT рятує нас, коли нас цікавить саме **середнє**. Але лог-трансформація все одно корисна — просто для **іншого** завдання: вона наближає **сам розподіл** даних до нормального, що потрібно, наприклад, для QQ-plot-діагностики, регресійних моделей або коли ми хочемо описати **типове** (а не середнє) значення через медіану/геометричне середнє.

Подивимось, що з формою розподілу робить логарифм:

In [ ]:
df['price_log'] = np.log(df.price)
df.price_log.hist(bins=100)

In [ ]:
stats.probplot(df.price_log, dist='norm', plot=plt)
plt.title('QQ-plot: log(price)')
plt.show()

Значно краще: гістограма симетричніша, а QQ-plot лежить набагато ближче до прямої — `log(price)` явно ближчий до нормального розподілу, ніж сама `price`.

Тепер — **головна пастка**, на якій легко помилитися. Побудуємо ДІ в лог-просторі й повернемо його назад у долари:

In [ ]:
mean_log = df.price_log.mean()
std_log = df.price_log.std()

ci_log = stats.norm.interval(0.95, loc=mean_log, scale=std_log / np.sqrt(n))

# повертаємось у долари: np.exp — правильна обернена функція до np.log
ci_log_dollars = np.exp(ci_log[0]), np.exp(ci_log[1])
ci_log_dollars

Порівняймо дві відповіді на те саме питання 5:

| Підхід | 95% ДІ |
|---|---|
| ДІ напряму по `price` (CLT) | ≈ (535 194, 544 983) |
| ДІ по `log(price)`, повернутий через `exp` | ≈ (460 835, 467 352) |

Різниця — **понад 70 000 $ (≈14%)**, і це не похибка округлення. Причина: `exp(mean(log(x)))` — це **геометричне середнє** `x`, а не арифметичне. Для правостороннього скошеного розподілу (як наш `price`) геометричне середнє завжди нижче за арифметичне й ближче до медіани (`450 000`).

**Висновок, який варто запам'ятати:** довірчий інтервал, побудований у log-просторі й повернутий назад експонентою, оцінює **геометричне середнє / типове значення**, а не арифметичне середнє. Це не "неправильний" результат — просто відповідь на **інше** питання. Якщо бізнес-питання буквально "яке середнє" — рахуйте ДІ напряму (нас рятує CLT); якщо питання ближче до "яка типова ціна" — log-простір і геометричне середнє доречніші, і до речі, ближчі до медіани, яку ми обговорювали ще в лекції про розподіли.

### Чому медіана не потрапляє в жоден із цих двох інтервалів — і це нормально

Зведемо всі числа, які ми тут отримали, в одну таблицю:

| Величина | Значення |
|---|---|
| Арифметичне середнє (`price.mean()`) | 540 088 usd |
| 95% ДІ арифметичного середнього (CLT, по сирій `price`) | (535 194, 544 983) |
| Геометричне середнє (`exp` від середнього в лог-просторі) | 464 082 usd |
| 95% ДІ геометричного середнього (лог-простір → `exp`) | (460 835, 467 352) |
| Медіана (`price.median()`) | 450 000 usd |

Медіана (450 000) не потрапляє в жоден з двох інтервалів — ані в (535 194, 544 983), ані в (460 835, 467 352). На перший погляд це виглядає як помилка, але це не так: **кожен довірчий інтервал побудований навколо своєї конкретної точкової оцінки й оцінює лише невизначеність саме цієї оцінки**, а не "правильний діапазон цін" загалом.

* ДІ `(535 194, 544 983)` каже: "ми дуже впевнені, що **арифметичне середнє** генеральної сукупності лежить саме тут" — і це дійсно так, адже середнє й так дорівнює 540 088, а не 450 000.
* ДІ `(460 835, 467 352)` каже те саме, але про **геометричне середнє** (464 082) — і теж не про медіану.
* Медіана (450 000) — це взагалі третя, окрема точкова оцінка. У неї є власний довірчий інтервал (його рахують інакше, іншими методами — наприклад, бутстрепом), і цей інтервал був би вузьким **навколо 450 000**, а не навколо 540 088 чи 464 082.

Вузькість обох ДІ показує не те, що ми "влучили" в якесь універсальне правильне число, а те, що при `n = 21613` ми дуже **точно** оцінили ту конкретну величину (середнє чи геометричне середнє), яку рахували. Те, що середнє, геометричне середнє й медіана — три різні числа з різними довірчими інтервалами навколо кожного, — якраз і є наслідком скошеності розподілу `price`, яку ми бачили на гістограмі й QQ-plot на початку цього розділу.

### Підсумок відповіді на питання 5

* Середня ціна будинку в сегменті з 95% впевненістю лежить у діапазоні **≈ 535 200 – 544 980 usd**.
* Якщо ж цікавить саме "типова" ціна (менш чутлива до дорогих викидів) — **≈ 460 800 – 467 350 usd**, і це ближче до медіани (`450 000 sud`).
* Медіана сама по собі (`450 000 usd`) не входить в жоден з цих двох інтервалів — і не повинна: це третя окрема величина зі своєю власною невизначеністю, а не "правильна відповідь", яку мали б "знайти" два попередні ДІ.
* Усі числа коректні — вони просто відповідають на різні версії одного й того ж бізнес-питання, і аналітик повинен явно вказувати, яке саме число він має на увазі, коли каже "середня ціна".